In [16]:
! pip install shapely

Import libraries :

In [17]:
import pandas as pd
from shapely.geometry import Point, shape
import json

Open the files :

In [18]:
df = pd.read_csv("../flickr_data2.csv")

with open("Lyon.geojson") as f:
    lyon_geojson = json.load(f)
lyon_polygon = shape(lyon_geojson['geometry'])  

C:\Users\Pyta\AppData\Local\Temp\ipykernel_31244\1561162387.py:1: DtypeWarning: Columns (11,12) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../flickr_data2.csv")


Test if the (long, lat) is located inside the polygon which defines Lyon area :

In [19]:
def is_inside(row):
    point = Point(row[' long'], row[' lat'])
    return lyon_polygon.contains(point)

In [20]:
df_inside = df[df.apply(is_inside, axis=1)]
df_inside.to_csv("flickr_data_geo_cleaned.csv", index=False)

Print the results :

In [21]:
print(f"Number of pictures in the original csv : {len(df)}, Pictures taken in Lyon : {len(df_inside)}")

Number of pictures in the original csv : 420240, Pictures taken in Lyon : 316440


# Text Processing


Get Tags

In [22]:
clean_df = pd.read_csv("./flickr_data_geo_cleaned.csv")

print(clean_df[' tags'])
print(len(clean_df[' tags']))

0                        chair,lyon,rhône,chaise,rhônealpes
1                                                       NaN
2                                                365,iphone
3                     poste,lyon,streetphotography,rue,gens
4                           lyon,streetphotography,rue,gens
                                ...                        
316435    europe,france,lyon,croixrousse,streetart,wheat...
316436                                                  NaN
316437    auvergnerhônealpes,rhône,lyonnais,valléedurhôn...
316438    auvergnerhônealpes,rhône,lyonnais,valléedurhôn...
316439    ngc,lyon,paysage,landscape,ville,urbain,town,tour
Name:  tags, Length: 316440, dtype: object
316440


C:\Users\Pyta\AppData\Local\Temp\ipykernel_31244\68963168.py:1: DtypeWarning: Columns (11,12) have mixed types. Specify dtype option on import or set low_memory=False.
  clean_df = pd.read_csv("./flickr_data_geo_cleaned.csv")


Number of potentials tags

In [39]:

import re
nona_df = clean_df[' tags'].dropna()
count = nona_df.apply(lambda x: len(re.findall(r'[^\W\d_]++', str(x)))).sum()

print(f"Number of tags in the cleaned dataset: {count}")
print(nona_df)

Number of tags in the cleaned dataset: 2263388
0                        chair,lyon,rhône,chaise,rhônealpes
2                                                365,iphone
3                     poste,lyon,streetphotography,rue,gens
4                           lyon,streetphotography,rue,gens
5         lyon,streetphotography,rue,montblanc,gens,mont...
                                ...                        
316434    europe,france,lyon,croixrousse,streetart,wall,...
316435    europe,france,lyon,croixrousse,streetart,wheat...
316437    auvergnerhônealpes,rhône,lyonnais,valléedurhôn...
316438    auvergnerhônealpes,rhône,lyonnais,valléedurhôn...
316439    ngc,lyon,paysage,landscape,ville,urbain,town,tour
Name:  tags, Length: 234929, dtype: object


In [ ]:
idf={}  # Inverse Document Frequency dictionary
total_docs = len(nona_df)
filtered_tags=["aplusphoto","view","fuime","night","chaise","chair","iphone","streetphotography","rue","gens","nuit","francia","poste","river","notte","night"]
for tags in nona_df:
    unique_tags = set(re.findall(r'[^\W\d_]+', str(tags)))
    
    unique_tags = {tag.lower() for tag in unique_tags if tag.lower() not in filtered_tags}  # Normalize to lowercase
    for tag in unique_tags:
        idf[tag] = idf.get(tag, 0) + 1
for tag in idf:
    idf[tag] = total_docs / idf[tag]
print("IDF values for tags:")
for tag, value in idf.items():
    print(f"{tag}: {value}")
    


    

IDF values for tags:
rhône: 9.780150701469548
lyon: 1.3393365145062626
rhônealpes: 13.519537319445243
montblanc: 2397.234693877551
groscailloux: 58732.25
montagnes: 39154.833333333336
croixrousse: 18.315194511577143
france: 2.312407106648949
lesphotosdevoyage: 19577.416666666668
fourvière: 35.1374513909662
fiume: 1312.4525139664804
lione: 41.28079423651379
ponte: 1365.8662790697674
bridge: 78.28357214261912
pont: 80.18054607508532
panorama: 172.7419117647059
fleuve: 108.06301747930083
saone: 77.40658978583195
aplusphoto: 33561.28571428572
flickrdiamond: 14683.0625
platinumheartaward: 21357.18181818182
theunforgettablepictures: 14683.0625
impressedbeauty: 46985.8
blueribbonwinner: 33561.28571428572
goldstaraward: 21357.18181818182
mywinners: 4606.450980392156
theperfectphotographer: 58732.25
goldstarawardgoldmedalwinner: 58732.25
betterthangood: 23492.9
artofimages: 9788.708333333334
platinumphoto: 23492.9
soe: 11187.095238095239
naturesfinest: 33561.28571428572
rendezvous: 58732.25
sup

In [74]:
#Cleaning by the lower IDF values

threshold = 18  # Default threshold
filtered_tags = []

idf = {tag: value for tag, value in idf.items() if value >= threshold}
count= len(idf)
print(f"Number of tags after filtering with threshold {threshold}: {count}")
print("IDF values for tags:")
for tag, value in idf.items():
    print(f"{tag}: {value}")

Number of tags after filtering with threshold 18: 31757
IDF values for tags:
montblanc: 2397.234693877551
groscailloux: 58732.25
montagnes: 39154.833333333336
croixrousse: 18.315194511577143
lesphotosdevoyage: 19577.416666666668
fourvière: 35.1374513909662
fiume: 1312.4525139664804
lione: 41.28079423651379
ponte: 1365.8662790697674
bridge: 78.28357214261912
pont: 80.18054607508532
panorama: 172.7419117647059
fleuve: 108.06301747930083
saone: 77.40658978583195
aplusphoto: 33561.28571428572
flickrdiamond: 14683.0625
platinumheartaward: 21357.18181818182
theunforgettablepictures: 14683.0625
impressedbeauty: 46985.8
blueribbonwinner: 33561.28571428572
goldstaraward: 21357.18181818182
mywinners: 4606.450980392156
theperfectphotographer: 58732.25
goldstarawardgoldmedalwinner: 58732.25
betterthangood: 23492.9
artofimages: 9788.708333333334
platinumphoto: 23492.9
soe: 11187.095238095239
naturesfinest: 33561.28571428572
rendezvous: 58732.25
supershot: 10214.304347826086
golddragon: 58732.25
dia

In [68]:
idf_2={}  # Inverse Document Frequency dictionary
total_docs = len(nona_df)
print(f"Total documents for recalculating IDF: {total_docs}")
for tags in nona_df:
    unique_tags = set(re.findall(r'[^\W\d_]+', str(tags)))

    for tag in unique_tags:
        if tag in idf:
            idf_2[tag] = idf_2.get(tag, 0) + 1
for tag in idf_2:
    idf_2[tag] = total_docs / idf_2[tag]

print("IDF values for tags:")
for tag, value in idf_2.items():
    print(f"{tag}: {value}")

Total documents for recalculating IDF: 234929
IDF values for tags:
montblanc: 2397.234693877551
groscailloux: 58732.25
montagnes: 39154.833333333336
croixrousse: 18.315194511577143
lesphotosdevoyage: 19577.416666666668
fourvière: 35.1374513909662
saone: 77.40658978583195
fiume: 1312.4525139664804
lione: 41.28079423651379
ponte: 1365.8662790697674
bridge: 78.28357214261912
pont: 80.18054607508532
night: 32.51162468862441
panorama: 172.7419117647059
river: 54.5585229911751
fleuve: 108.06301747930083
notte: 1576.7046979865772
view: 347.52810650887574
aplusphoto: 33561.28571428572
flickrdiamond: 14683.0625
platinumheartaward: 21357.18181818182
impressedbeauty: 46985.8
theunforgettablepictures: 14683.0625
blueribbonwinner: 33561.28571428572
goldstaraward: 21357.18181818182
mywinners: 4606.450980392156
theperfectphotographer: 58732.25
goldstarawardgoldmedalwinner: 58732.25
betterthangood: 23492.9
artofimages: 9788.708333333334
platinumphoto: 23492.9
soe: 11187.095238095239
naturesfinest: 335